# 02 — Model Training

This notebook covers:
- Loading YOLOv8 pretrained weights
- Fine-tuning on the custom traffic dataset
- Hyperparameter configuration
- Training callbacks & early stopping
- Metrics: mAP50, mAP50-95, Precision, Recall
- Saving best weights for inference

In [1]:
# ── Imports ───────────────────────────────────────────────────────────
import os, yaml, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from ultralytics import YOLO

BASE_DIR   = Path(".")
DATA_YAML  = BASE_DIR / "data" / "processed" / "data.yaml"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

print("Ultralytics YOLO loaded ✓")
print(f"Dataset config: {DATA_YAML}")

Ultralytics YOLO loaded ✓
Dataset config: data\processed\data.yaml


## 2.1  Choose Base Model

| Model | Size | mAP50-95 (COCO) | Speed (ms/img) |
|-------|------|-----------------|----------------|
| YOLOv8n | 3.2 M | 37.3 | 0.99 |
| YOLOv8s | 11.2 M | 44.9 | 1.20 |
| YOLOv8m | 25.9 M | 50.2 | 1.83 |
| YOLOv8l | 43.7 M | 52.9 | 2.39 |
| YOLOv8x | 68.2 M | 53.9 | 3.53 |

**Recommendation:** Start with `yolov8s.pt` for a balance of speed and accuracy.

In [2]:
# ── Load pretrained YOLOv8s ───────────────────────────────────────────
MODEL_NAME = "yolov8s.pt"   # will auto-download on first run
model = YOLO(MODEL_NAME)
print(model.info())

YOLOv8s summary: 225 layers, 11166560 parameters, 0 gradients, 28.8 GFLOPs
(225, 11166560, 0, 28.816844800000002)


## 2.2  Training Configuration

In [3]:
TRAIN_CONFIG = dict(
    data         = str(DATA_YAML),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,           # reduce to 8 if GPU OOM
    lr0          = 0.01,         # initial learning rate
    lrf          = 0.01,         # final LR fraction (cosine schedule)
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    cos_lr       = True,
    augment      = True,         # built-in Mosaic, Mixup, colour jitter
    degrees      = 5.0,          # rotation augmentation
    fliplr       = 0.5,
    flipud       = 0.0,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    patience     = 20,           # early stopping patience
    save         = True,
    save_period  = 10,
    project      = str(MODELS_DIR),
    name         = "traffic_v1",
    exist_ok     = True,
    pretrained   = True,
    optimizer    = "AdamW",
    verbose      = True,
    seed         = 42,
    device       = "0",          # change to "cpu" if no GPU
    workers      = 4,
    amp          = True,         # automatic mixed precision (faster on GPU)
    close_mosaic = 10,           # disable mosaic in last N epochs
    # Class weights to handle imbalance:
    # cls          = 0.5,
)
print("Training config defined ✓")

Training config defined ✓


## 2.3  Train

In [4]:
# ── Start training ───────────────────────────────────────────────────
# NOTE: Ensure dataset.yaml and images/labels are in place before running.
# Remove the comment below to execute.

results = model.train(**TRAIN_CONFIG)
print("Training complete!")
print(f"Best weights saved → {results.save_dir}/weights/best.pt")

# ─── Simulate training result paths for demo ──────────────────────────
BEST_PT = MODELS_DIR / "traffic_v1" / "weights" / "best.pt"
LAST_PT = MODELS_DIR / "traffic_v1" / "weights" / "last.pt"
print(f"Expected best weights location: {BEST_PT}")

New https://pypi.org/project/ultralytics/8.4.34 available  Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.0  Python-3.10.11 torch-2.3.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=data\processed\data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=0, workers=4, project=models, name=traffic_v1, exist_ok=True, pretrained=True, optimizer=AdamW, verbose=True, seed=42, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_masks=False, embed=No

RuntimeError: Dataset 'data/processed/data.yaml' error  'data\processed\data.yaml' does not exist

## 2.4  Training Metrics Visualisation

In [ ]:
def plot_training_metrics(results_csv: str):
    """Parse Ultralytics results.csv and plot key metrics."""
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    metrics = [
        ("train/box_loss",   "Box Loss (train)"),
        ("train/cls_loss",   "Class Loss (train)"),
        ("val/box_loss",     "Box Loss (val)"),
        ("metrics/precision(B)", "Precision"),
        ("metrics/recall(B)",    "Recall"),
        ("metrics/mAP50(B)",     "mAP@0.50"),
    ]
    for ax, (col, title) in zip(axes.flat, metrics):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=1.5)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)

    plt.suptitle("YOLOv8 Training Metrics", fontsize=14)
    plt.tight_layout()
    plt.show()

# Uncomment after training:
plot_training_metrics(str(MODELS_DIR / "traffic_v1" / "results.csv"))

# ─── Show sample validation batch (if available) ─────────────────────
def show_val_batch(run_dir: str):
    for name in ["val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
        path = Path(run_dir) / name
        if path.exists():
            img = mpimg.imread(str(path))
            plt.figure(figsize=(12, 6))
            plt.imshow(img)
            plt.title(name)
            plt.axis("off")
            plt.show()

# show_val_batch(str(MODELS_DIR / "traffic_v1"))
print("Metric plotting helpers defined ✓")

## 2.5  Evaluate on Validation Set

In [ ]:
# ── Validation / mAP evaluation ──────────────────────────────────────
def evaluate_model(weights_path: str, data_yaml: str):
    model = YOLO(weights_path)
    metrics = model.val(
        data   = data_yaml,
        imgsz  = 640,
        batch  = 16,
        conf   = 0.25,
        iou    = 0.45,
        device = "0",
        verbose= True,
    )
    print("\n── Evaluation Results ──────────────────────────")
    print(f"  mAP50      : {metrics.box.map50:.4f}")
    print(f"  mAP50-95   : {metrics.box.map:.4f}")
    print(f"  Precision  : {metrics.box.mp:.4f}")
    print(f"  Recall     : {metrics.box.mr:.4f}")
    return metrics

# Uncomment after training:
# if BEST_PT.exists():
#     metrics = evaluate_model(str(BEST_PT), str(DATA_YAML))
print("evaluate_model() defined ✓")

## 2.6  Hyperparameter Tuning (Ray Tune integration)

In [ ]:
# ── Optional: Automatic Hyperparameter Search ─────────────────────────
# Ultralytics has built-in Ray Tune support.
# Install: pip install ray[tune] -q

def run_hparam_search(data_yaml: str, iterations: int = 10):
    """
    Run hyperparameter search using YOLO's built-in tuner.
    Each trial trains for a few epochs on a subset.
    """
    model = YOLO("yolov8s.pt")
    result = model.tune(
        data       = data_yaml,
        epochs     = 20,
        iterations = iterations,
        optimizer  = "AdamW",
        plots      = False,
        save       = False,
        val        = True,
    )
    print("Best hyperparameters:", result)
    return result

# Uncomment to run (time-consuming):
# best_hp = run_hparam_search(str(DATA_YAML), iterations=20)
print("Hyperparameter search helper defined ✓")

## 2.7  Export Model for Deployment

In [ ]:
# ── Export to ONNX / TensorRT for faster inference ───────────────────
def export_model(weights_path: str, fmt: str = "onnx"):
    """
    Supported formats: onnx, engine (TensorRT), coreml, ncnn, etc.
    """
    model = YOLO(weights_path)
    path  = model.export(format=fmt, imgsz=640, half=True)
    print(f"Exported model → {path}")
    return path

# Uncomment after training:
 if BEST_PT.exists():
     onnx_path = export_model(str(BEST_PT), fmt="onnx")
print("export_model() defined ✓")

## ✅ Notebook 2 Summary

- YOLOv8s loaded with pretrained COCO weights
- Training config with 100 epochs, AdamW, cosine LR, built-in augmentation
- Validation & mAP evaluation helpers
- Hyperparameter tuning via Ray Tune
- Model export to ONNX

**Next:** `03_object_tracking.ipynb`